In [1]:
import pandas as pd
import ast

# В этой тетрадке описан процесс выделения сессий из журнала логов
Это шаблон, по которому из журнала логов клиента можно выделить сессии для дальнейшего анализа


In [2]:
df_journal = pd.read_csv('data/synthetic_logs.csv')

In [3]:
# Recovering `stoppedAt` events without `actor.id` using `record`
# and sorting events by user action timeline

FINISH_EXAM = 'stoppedAt,status'
START_EXAM = 'startedAt,signedAt,conclusion,status'

df_journal['changes'] = df_journal['changes'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

base = df_journal[['record','actor.id','actor.role','changes']]
students_only = base.loc[base['actor.role'].eq('student') & base['actor.id'].notna(), ['record','actor.id']]

# Core logic: restoring `actor.id` for FINISH_EXAM events
# FINISH_EXAM events can be missing in some cases,
# e.g. when a user leaves the exam page and the session is closed automatically.
# If we observe that a given `record` contains one and only one valid `actor.id` in dataset,
# we can recover missing `actor.id` values for FINISH_EXAM events within the same record.
record_actor_stats = students_only.groupby('record', sort=False)['actor.id'].nunique()
records_with_single_student = record_actor_stats.index[record_actor_stats.eq(1)]

record_to_actor = (
    students_only.loc[students_only['record'].isin(records_with_single_student)]
    .groupby('record', sort=False)['actor.id']
    .first()
)

mask = base['actor.id'].isna() & base['changes'].eq(FINISH_EXAM) & base['record'].isin(record_to_actor.index)

df_journal_sessionazation = df_journal.copy()
df_journal_sessionazation.loc[mask, 'actor.id'] = base.loc[mask, 'record'].map(record_to_actor)

# Filtering dataset to include only student actions (including recovered FINISH_EXAM events)
df_journal_sessionazation = df_journal_sessionazation[df_journal_sessionazation['actor.role'].eq('student')]

In [4]:
# Creating additional features based on temporal distance between consecutive events
# These features are used to detect new attempts

df_journal_sessionazation['createdAt'] = pd.to_datetime(df_journal_sessionazation['createdAt'], unit='ms')
df_journal_sessionazation = df_journal_sessionazation.sort_values(['actor.id', 'createdAt'])

df_journal_sessionazation['next_event_time'] = df_journal_sessionazation.groupby('actor.id')['createdAt'].shift(-1)
df_journal_sessionazation['duration_till_next_event'] = (df_journal_sessionazation['next_event_time'] - df_journal_sessionazation['createdAt']).dt.total_seconds() / 60

df_journal_sessionazation['duration_since_last_event'] = df_journal_sessionazation.groupby('actor.id')['duration_till_next_event'].shift()

df_journal_sessionazation['last_event'] = (df_journal_sessionazation.groupby('actor.id')['changes'].shift())

large_gap = df_journal_sessionazation['duration_since_last_event'] >= 10

# Key assumption: a new attempt is always created if the inactivity gap exceeds 60 minutes,
# even if no explicit termination event exists.
# (This threshold may need adjustment depending on exam duration and configuration.)
large_super_gap = df_journal_sessionazation['duration_since_last_event'] >= 60

after_stop = df_journal_sessionazation['last_event'].apply(
    lambda x: any(event in FINISH_EXAM for event in x) if isinstance(x, list) else False)

is_start = df_journal_sessionazation['changes'].apply(
    lambda x: any(event in START_EXAM for event in x) if isinstance(x, list) else False)

is_stop = df_journal_sessionazation['changes'].apply(
    lambda x: any(event in FINISH_EXAM for event in x) if isinstance(x, list) else False)

df_journal_sessionazation['exam_state'] = (is_start.astype(int) - is_stop.astype(int))

df_journal_sessionazation['exam_active'] = (
    df_journal_sessionazation
        .groupby('actor.id')['exam_state']
        .cumsum()
        .gt(0)
)

df_journal_sessionazation['new_attempt'] = (
    after_stop |
    (large_gap & ~df_journal_sessionazation['exam_active']) |
    large_super_gap
)

In [5]:
# Refining attempt duration logic and filtering only valid attempts
# A fallback rule is applied to handle cases where FINISH_EXAM events are not fully recovered.

forced_stop = large_super_gap.astype(int)

df_journal_sessionazation['exam_state_corrected'] = (
    is_start.astype(int)
    - is_stop.astype(int)
    - forced_stop
)

df_journal_sessionazation['exam_active'] = (
    df_journal_sessionazation
        .groupby('actor.id')['exam_state_corrected']
        .cumsum()
        .gt(0)
)

df_journal_sessionazation.loc[df_journal_sessionazation['duration_since_last_event'].isna(), 'new_attempt']= True

df_journal_sessionazation['attempt_id'] = (
    df_journal_sessionazation.groupby('actor.id')['new_attempt']
    .cumsum()
)

In [6]:
# Some logs contain isolated or single events after exam completion or at random timestamps.
# These are not valid attempts and are treated as noise or link-check behavior.
# We remove such cases to avoid distortion of failed attempt statistics.
# A valid attempt must contain at least 2 events.
attempt_sizes = (
    df_journal_sessionazation
    .groupby(['actor.id', 'attempt_id'])
    .size()
    .rename('attempt_size')
    .reset_index()
)

df_journal_sessionazation = df_journal_sessionazation.merge(
    attempt_sizes,
    on=['actor.id', 'attempt_id'],
    how='left'
)

df_journal_sessionazation['is_start'] = is_start
attempt_has_exam = (
    df_journal_sessionazation
    .groupby(['actor.id', 'attempt_id'])['is_start']
    .max()
    .rename('attempt_has_exam')
    .reset_index()
)

df_journal_sessionazation = df_journal_sessionazation.merge(
    attempt_has_exam,
    on=['actor.id', 'attempt_id'],
    how='left'
)

df_journal_sessionazation['valid_attempt'] = (
    (df_journal_sessionazation['attempt_size'] > 2)
    | (df_journal_sessionazation['attempt_has_exam'])
    | (df_journal_sessionazation['changes'] == 'startedAt,signedAt,conclusion,status')
)

df_journal_sessionazation = (
    df_journal_sessionazation[
        df_journal_sessionazation['valid_attempt']].copy()
)

df_journal_sessionazation['attempt_id'] = (
    df_journal_sessionazation
        .groupby('actor.id')['attempt_id']
        .rank(method='dense')
        .astype(int)
)

In [7]:
# Adding derived metrics:
# - time spent on onboarding / access preparation
# - attempt success flag
# - number of errors per attempt

attempt_start_time = (
    df_journal_sessionazation[df_journal_sessionazation['new_attempt']]
        .groupby(['actor.id', 'attempt_id'])['createdAt']
        .min()
        .rename('attempt_start_time'))

df_journal_sessionazation = df_journal_sessionazation.merge(
    attempt_start_time,
    on=['actor.id', 'attempt_id'],
    how='left')

df_journal_sessionazation['is_exam_start'] = df_journal_sessionazation['changes'].apply(
    lambda x: any(event in START_EXAM for event in x) if isinstance(x, list) else False)

exam_start_time = (
    df_journal_sessionazation[df_journal_sessionazation['is_exam_start']]
        .groupby(['actor.id', 'attempt_id'])['createdAt']
        .min())

df_journal_sessionazation = df_journal_sessionazation.merge(
    exam_start_time.rename('exam_start_time'),
    on=['actor.id', 'attempt_id'],
    how='left')

attempt_end_time = (
    df_journal_sessionazation
        .groupby(['actor.id', 'attempt_id'])['createdAt']
        .max()
        .rename('attempt_end_time'))

df_journal_sessionazation = df_journal_sessionazation.merge(
    attempt_end_time,
    on=['actor.id', 'attempt_id'],
    how='left')

df_journal_sessionazation['equipment_check_end'] = (
    df_journal_sessionazation['exam_start_time']
        .fillna(df_journal_sessionazation['attempt_end_time']))

df_journal_sessionazation['equipment_check_time_min'] = (
    df_journal_sessionazation['equipment_check_end']
    - df_journal_sessionazation['attempt_start_time']
).dt.total_seconds() / 60

df_journal_sessionazation['is_success_event'] = df_journal_sessionazation['changes'].apply(
    lambda x: any(event in START_EXAM for event in x) if isinstance(x, list) else False)


df_journal_sessionazation['attempt_success'] = (
    df_journal_sessionazation.groupby(['actor.id', 'attempt_id'])['is_success_event']
        .transform('max')
        .astype(int))


ERROR = {"error"}

df_journal_sessionazation['is_error'] = df_journal_sessionazation['changes'].apply(
    lambda x: any(event in ERROR for event in x) if isinstance(x, list) else False)

df_journal_sessionazation['amount_of_errors'] = df_journal_sessionazation.groupby(['actor.id', 'attempt_id'])['is_error'].transform('sum').astype(int)

In [8]:
agg_map = {

    'createdAt': 'min',
    'attempt_end_time': 'first',

    'actor.id': 'first',

    'equipment_check_time_min': 'min',
    'attempt_success': 'max',
    'attempt_id': 'first',
    'attempt_size' : 'first',
    'amount_of_errors': 'first',

    'browser': 'first',
    'os': 'first',
    'is_mobile': 'first',
    'browser_version': 'first',
    'os_version': 'first',

    'country': 'first',
    'region': 'first',

}

attempts_df = (
    df_journal_sessionazation
        .groupby(['actor.id', 'attempt_id'], as_index=False)
        .agg(agg_map)
)

In [9]:
# Keeping only top-3 most frequent error types for analysis clarity
top_errors = (
    df_journal_sessionazation['error'].value_counts().head(3).index)

error_counts = (
    df_journal_sessionazation[df_journal_sessionazation['error'].notna()]
    .assign(error=lambda x: x['error'].where(
        x['error'].isin(top_errors),
        'other'
    ))
    .groupby(['actor.id', 'attempt_id', 'error'])
    .size()
    .unstack(fill_value=0)
)

attempts_df = attempts_df.merge(
    error_counts,
    on=['actor.id', 'attempt_id'],
    how='left'
)

error_cols = error_counts.columns
attempts_df[error_cols] = attempts_df[error_cols].fillna(0)

error_cols = list(error_counts.columns)
cols = list(attempts_df.columns)
insert_pos = cols.index('amount_of_errors') + 1

new_cols = (
    cols[:insert_pos] +
    error_cols +
    [col for col in cols[insert_pos:] if col not in error_cols]
)
attempts_df = attempts_df[new_cols]


In [10]:
# Final dataset validation check
attempts_df

,createdAt,attempt_end_time,actor.id,equipment_check_time_min,attempt_success,attempt_id,attempt_size,amount_of_errors,Error: Face not found,Error: Incorrect facial position,Error: Incorrect passport position,other,browser,os,is_mobile,browser_version,os_version,country,region
0,2025-04-06 13:38:56.887000+00:00,2025-04-06 13:51:07.273000+00:00,user_00001,12.173100,1,1,8,2,0.0,0.0,0.0,1.0,Chrome,Windows,False,134.0.0,10,RU,Irkutsk Oblast
1,2025-04-06 13:58:11.873000+00:00,2025-04-06 13:58:11.873000+00:00,user_00001,0.000000,1,2,1,0,0.0,0.0,0.0,0.0,Chrome,Windows,False,134.0.0,10,RU,Irkutsk Oblast
2,2025-04-21 06:35:31.362000+00:00,2025-04-21 06:39:11.843000+00:00,user_00001,3.674683,1,3,4,0,0.0,0.0,0.0,0.0,Chrome Mobile iOS,iOS,True,135.0.7049,18.3.2,RU,Irkutsk Oblast
3,2025-04-21 06:47:12.466000+00:00,2025-04-21 06:51:14.418000+00:00,user_00001,4.032533,1,4,4,0,0.0,0.0,0.0,0.0,Chrome Mobile iOS,iOS,True,135.0.7049,18.3.2,RU,Irkutsk Oblast
4,2025-05-16 07:45:36.872000+00:00,2025-05-16 07:49:38.955000+00:00,user_00001,4.034717,0,5,6,3,1.0,1.0,0.0,0.0,Chrome,Windows,False,136.0.0,10,RU,Irkutsk Oblast
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
950,2026-04-06 07:38:42.983000+00:00,2026-04-06 07:42:25.982000+00:00,user_00548,3.716650,1,2,5,0,0.0,0.0,0.0,0.0,Yandex Browser,Windows,False,24.10.0,10,RU,Irkutsk Oblast
951,2026-04-06 07:48:16.572000+00:00,2026-04-06 07:55:25.299000+00:00,user_00549,7.145450,1,1,7,2,0.0,0.0,0.0,1.0,Yandex Browser,Windows,False,26.3.0,10,RU,NaN
952,2026-04-06 08:01:47.369000+00:00,2026-04-06 08:01:47.369000+00:00,user_00549,0.000000,1,2,1,0,0.0,0.0,0.0,0.0,Yandex Browser,Windows,False,26.3.0,10,RU,NaN
953,2026-04-06 07:57:05.807000+00:00,2026-04-06 08:00:58.081000+00:00,user_00550,3.871233,1,1,5,0,0.0,0.0,0.0,0.0,Yandex Browser,Windows,False,26.3.0,10,RU,Transbaikal Territory


In [11]:
attempts_df.to_csv('data/attempts_synth.csv', index=False)